# Big Data Analytics Lab 1
## Function-centric programming in Python

Goal for today:
- Practice a function-centric style of programming in Python
- Understand the difference between imperative, OOP, and functional approaches
- Use map, filter, reduce, and related tools on a real dataset
- Build intuition for lazy evaluation, pure functions, and immutability

How to use this notebook:
- Run cells top to bottom
- Where you see `# TODO`, write your own code
- Each section has a short example followed by a small exercise
- Focus on understanding the pattern, not just getting the right answer

Dataset
- Wikipedia movie plots (one row per movie)
- We will use the plot text plus a few metadata fields to make examples concrete

## 1) What is functional programming?

Functional programming (FP) is a programming style where you:
- treat computation as applying functions to data
- avoid changing (mutating) data in place when possible
- build programs as small transformation steps that can be combined

In FP, you often work with sequences (lists, iterators, rows of a dataset) and specify:
- which transformation to apply to each item (map-like)
- which items to keep (filter-like)
- how to summarize many items into fewer results (reduce-like, or group-and-aggregate)

In object-oriented programming (OOP), it is common to:
- represent data as objects
- attach behavior to objects via methods (object.method())
- manage internal state inside objects

Python supports both styles.
In this course, we practice a function-centric style because it scales well from:
- small lists in Python
to
- very large datasets processed in chunks, streams, or distributed systems.

___

This notebook was prepared using the below resources. If you're interested you can reach out to receive a copy:
1. Fluent Python by Luciano Ramalho.
2. Functional Programming in Python, from the Python docs.

___

In [1]:
import re
from pathlib import Path
import pandas as pd

DATA_PATH = "wiki_movie_plots_deduped.csv"

# If you keep the CSV elsewhere, update DATA_PATH.
# Example:
# DATA_PATH = Path.home() / "Downloads" / "wiki_movie_plots_deduped.csv"



df_raw = pd.read_csv(DATA_PATH)
df_raw.shape


(34886, 8)

In [2]:
df_raw.columns

Index(['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast',
       'Genre', 'Wiki Page', 'Plot'],
      dtype='object')

In [3]:
# Column names in the Kaggle dataset
YEAR_COL = "Release Year"
TITLE_COL = "Title"
ORIGIN_COL = "Origin/Ethnicity"
PLOT_COL = "Plot"
DIRECTOR_COL = "Director"
CAST_COL = "Cast"
GENRE_COL = "Genre"


# Quick sanity check (prints, no assertions)
print("Columns:", list(df_raw.columns))
print("Rows:", len(df_raw))
df_raw[[YEAR_COL, TITLE_COL, ORIGIN_COL, PLOT_COL, DIRECTOR_COL, CAST_COL, GENRE_COL]].head(3)

Columns: ['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot']
Rows: 34886


,Release Year,Title,Origin/Ethnicity,Plot,Director,Cast,Genre
0,1901,Kansas Saloon Smashers,American,"A bartender is working at a saloon, serving dr...",Unknown,NaN,unknown
1,1901,Love by the Light of the Moon,American,"The moon, painted with a smiling face hangs ov...",Unknown,NaN,unknown
2,1901,The Martyred Presidents,American,"The film, just over a minute long, is composed...",Unknown,NaN,unknown


We will often work with a “sequence of records” view of the dataset.

A simple Python-friendly representation is:
- one movie = one dictionary
- many movies = a list of dictionaries

This makes the examples look like “transform a sequence”, which is the mindset we want.

In [4]:
movies = df_raw[[YEAR_COL, TITLE_COL, ORIGIN_COL, PLOT_COL, DIRECTOR_COL, CAST_COL, GENRE_COL]].to_dict("records")
movies[:2]

[{'Release Year': 1901,
  'Title': 'Kansas Saloon Smashers',
  'Origin/Ethnicity': 'American',
  'Plot': "A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man, pulling his hat over his eyes and then dumping the beer over his head. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender then sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.[1]",
  'Director': 'Unknown',
  'Cast': nan,
  'Genre': 'unknown'},
 {'Release Year': 1901,
  'Title': 'Love by the Light of the Moon',
  'Origin/Ethnicity': 'American',
  'Plot': "The moon, painted with a smiling face hangs over a park at night. A young couple walking past a fence learn on a railing and look up. The moon smiles. They embrace, and the moon's smile gets bigger. They then sit do

## 2) Programming paradigms: imperative vs OOP vs functional

To see the difference, we will do a very simple task:

Task
- Count how many movies have Origin/Ethnicity equal to "American"

We will implement this task in three ways:
1) Imperative (explicit loop and counter)
2) Object-oriented (wrap data in a class, use a method)
3) Functional (express a transformation over a sequence)

> Don't worry if you can't follow the code as the goal here is not to understand the code, but rather the logic of it, i.e, how are we implementing said task?

In [ ]:
# 2.1 Imperative style: explicit loop + mutable counter

count = 0
for m in movies:
    if m[ORIGIN_COL] == "American":
        count += 1

print("Imperative count:", count)

In [ ]:
# 2.2 Object-oriented style: wrap the dataset in a class

class MovieDataset:
    def __init__(self, movies):
        self.movies = movies

    def count_origin(self, origin):
        count = 0
        for m in self.movies:
            if m[ORIGIN_COL] == origin:
                count += 1
        return count

ds = MovieDataset(movies)
print("OOP count:", ds.count_origin("American"))

In [ ]:
# 2.3 Functional style: specify the transformation on the sequence

# Option A: list comprehension + sum
count_fp_a = sum([1 for m in movies if m[ORIGIN_COL] == "American"])
print("Functional count (list comprehension):", count_fp_a)

# Option B: generator expression + sum (no intermediate list needed)
count_fp_b = sum(1 for m in movies if m[ORIGIN_COL] == "American")
print("Functional count (generator):", count_fp_b)

What should you notice?

- The imperative and OOP versions describe how to count (step-by-step).
- The functional versions focus on what we want:
  “sum 1 for each movie that matches a condition”.

This “what, not how” style becomes valuable when:
- you chain many steps (filter, transform, filter, transform)
- you want to swap steps easily
- you want to test steps separately

## 3) Why learn functional programming for big data?

When datasets become large, you often cannot:
- load everything comfortably into memory
- compute everything eagerly without wasting time

In large-scale data processing, two ideas show up repeatedly:

Immutability: 
> This means you avoid changing data in place and instead create new data structures with the desired changes.
- Prefer producing new results instead of changing shared data in place.
- This reduces surprises when work is parallelized or retried.

Lazy evaluation

> This means you specify transformations without executing them immediately.

- Build a plan of transformations first.
- Execute only when you request results.
- This lets systems optimize work and avoid unnecessary computation.

Even in plain Python, you can practice the same habits:
- small functions
- clear transformations
- minimal side effects

## 4) Pure functions vs impure functions

A pure function:
- uses only its explicit inputs
- returns a result
- does not change anything outside the function (no side effects)

An impure function:
- depends on external state (a global variable, the current time, a file)
- or changes external state (modifies a global, prints, writes to disk)
- or mutates its inputs in place

Pure functions are easier to:
- test
- reuse
- combine into pipelines

In [5]:
# 4.1 Pure function (very simple)

def add_vat(price, vat_rate):
    return price * (1 + vat_rate)

print(add_vat(10, 0.23))
print(add_vat(10, 0.23))  # same input, same output

12.3
12.3


In [22]:
# 4.2 Impure function (very simple): depends on external state

VAT_RATE = 0.23

def add_vat_impure(price):
    # depends on VAT_RATE which is not an explicit input
    return price * (1 + VAT_RATE)

print(add_vat_impure(10))

VAT_RATE = 0.10  # external state changed
print(add_vat_impure(10))  # same input, different output because VAT_RATE changed

12.3
11.0


## 5) Higher-order functions

A higher-order function is a function that:
- takes a function as input, or
- returns a function

Python uses this idea everywhere, for example:
- map(function, iterable)
- filter(function, iterable)
- sorted(iterable, key=function)

We start with small lists, then apply the same pattern to the movie dataset.

### 5.1 Higher-order function on a simple list

In [23]:

nums = [1, 2, 3, 4, 5]

def apply_to_all(f, xs):
    """Applies function f to each element of xs and returns a new list of results."""
    out = []
    for x in xs:
        out.append(f(x))
    return out

def double(x):
    return 2 * x

print(apply_to_all(double, nums))

[2, 4, 6, 8, 10]


### 5.2 The same idea using Python built-ins: map and filter
- `map` applies a function to each element of a sequence and returns an iterator of results.
- `filter` applies a predicate function to each element and returns an iterator of those that satisfy the predicate

In [24]:

def is_even(x):
    return x % 2 == 0

print(list(map(double, nums)))
print(list(filter(is_even, nums)))

[2, 4, 6, 8, 10]
[2, 4]


In [25]:
# 5.3 Higher-order functions on the dataset

def is_american(movie):
    return movie[ORIGIN_COL] == "American"

def title(movie):
    return movie[TITLE_COL]

american_titles = list(map(title, filter(is_american, movies)))
print("Number of American movies:", len(american_titles))
print("First 10 titles:", american_titles[:10])

Number of American movies: 17377
First 10 titles: ['Kansas Saloon Smashers', 'Love by the Light of the Moon', 'The Martyred Presidents', 'Terrible Teddy, the Grizzly King', 'Jack and the Beanstalk', 'Alice in Wonderland', 'The Great Train Robbery', 'The Suburbanite', 'The Little Train Robbery', 'The Night Before Christmas']


## 6) Lambda functions (anonymous functions)

A lambda function is a small function written in an expression form.

Syntax
- lambda inputs: expression

Important constraints in Python
- it is a single expression (no multi-line body)
- it returns the value of that expression

When lambda is useful
- short throwaway functions, especially as arguments to map/filter/sorted

When to avoid lambda
- if the logic needs a clear name
- if it is long enough to become unreadable

In [ ]:
# 6.1 Lambda: the simplest form (no dataset yet)

add1 = lambda x: x + 1
print(add1(10))

multiply = lambda a, b: a * b
print(multiply(6, 7))

In [ ]:
# 6.2 Lambda with map/filter on a small list

nums = [1, 2, 3, 4]

print(list(map(lambda x: x + 1, nums)))
print(list(filter(lambda x: x % 2 == 0, nums)))

In [11]:
# 6.3 Lambda with map/filter on the movie dataset

# Lambda 1: extract the title and release year as a formatted string
title_year = list(map(lambda m: f"{m[TITLE_COL]} ({m[YEAR_COL]})",movies[:5]))
print("Title + Year:", title_year)



Title + Year: ['Kansas Saloon Smashers (1901)', 'Love by the Light of the Moon (1901)', 'The Martyred Presidents (1901)', 'Terrible Teddy, the Grizzly King (1901)', 'Jack and the Beanstalk (1902)']


In [12]:
# Lambda 2: filter movies directed by a specific director
kubrick_movies = list(filter(lambda m: "Kubrick" in str(m[DIRECTOR_COL]),movies))
print("Kubrick movies:", [m[TITLE_COL] for m in kubrick_movies])

Kubrick movies: ['Fear and Desire', "Killer's Kiss", 'The Killing', 'Paths of Glory', 'Spartacus', 'Lolita', 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb', '2001: A Space Odyssey', 'Barry Lyndon', 'The Shining', 'Full Metal Jacket', 'Eyes Wide Shut', 'Dr. Strangelove', 'A Clockwork Orange']


### Exercise: write your own lambda functions using the dataset

Task A
- Create a lambda called is_bollywood that returns True only when Origin/Ethnicity == "Bollywood".
- Then compute how many Bollywood movies exist in the dataset.

Task B
- Create a lambda called mentions_magic that returns True when the plot contains the word "magic" (case-insensitive).
- Then list 10 movie titles whose plots mention "magic".

These are intentionally small: focus on writing the lambda correctly.

In [ ]:

bollywood_movies = list(filter(lambda m: "Bollywood" in str(m[ORIGIN_COL]), movies)) 
print("Bollywood movies:", len(bollywood_movies))

Bollywood movies: 2931


In [14]:
# Exercise B (TODO)

magic_movies = list(filter(lambda m: "Magic" in str(m[TITLE_COL]), movies)) # TODO: replace with a lambda

# Print up to 10 titles
print([m[TITLE_COL] for m in magic_movies[:10]])

['The Magic Cup', 'The Magician', 'Chandu the Magician', 'Music Is Magic', "Dr. Ehrlich's Magic Bullet", 'Black Magic', 'Magic Town', 'Black Magic', 'The Magic Carpet', 'Magical Maestro']



`filter` already provides the if/else so your lambda only needs to answer True or False.

Think of it this way:

| Part | Role |
|------|------|
| `filter(...)` | "keep this item IF..." |
| `lambda m:` | "given one movie m..." |
| `m[ORIGIN_COL] == "Bollywood"` | "...does this condition hold?" |

`filter` internally does the equivalent of:
```python
for m in movies:
    if lambda_returns_true(m):   # ← filter handles this
        keep(m)
```



In [20]:
from functools import reduce, partial, lru_cache
from operator import add
import itertools

## 7) map

map applies a function to every element in a sequence.

Conceptually:
- input:  [x1, x2, x3, ...]
- output: [f(x1), f(x2), f(x3), ...]

In Python, map does not create a list immediately. It returns an iterator.
To see the results, you typically wrap it with list(...), or you loop over it.

A very common alternative is a list comprehension:
- map(f, xs) is similar to [f(x) for x in xs]

In [ ]:
nums = [1, 2, 3, 4]

plus_one = map(lambda x: x + 1, nums)
print("map returns:", type(plus_one))
print(list(plus_one))

In [ ]:
# Example: extract titles from the dataset

titles_iter = map(lambda m: m[TITLE_COL], movies[:10])
print(list(titles_iter))

### Exercise: map on the movie dataset

Task
- Create an iterator of release years for the first 20 movies using map
- Convert it to a list and print it
- BONUS: keep only unique years and print that list
Hint
- Use YEAR_COL

In [ ]:

years_iter = map(lambda m: m[YEAR_COL], movies[:20])  # First 20 movies

years_list = list(years_iter) if years_iter is not None else None # this returns all the years for each movies release
years_iter = map(lambda m: m[YEAR_COL], movies[:20])  # First 20 movies
years_list_set = list(set(years_iter)) if years_iter is not None else None # if we want only the years, i.e, keeping only unique years we can use the set()
print(years_list)
print(years_list_set)

[1901, 1901, 1901, 1901, 1902, 1903, 1903, 1904, 1905, 1905, 1906, 1906, 1906, 1907, 1907, 1907, 1908, 1908, 1908, 1908]
[1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908]


## 9) reduce

reduce repeatedly combines items into a single result.

Conceptually:
- start with the first two items, combine them
- then combine that result with the next item
- continue until one value remains

In Python, reduce lives in functools, not as a built-in.

For summation, Python has sum(...), which is usually clearer.
Other “reducing” built-ins include, but not limited to, are:
- any(...)
- all(...)

In [ ]:
nums = [1, 2, 3, 4]

print("reduce add:", reduce(add, nums))
print("sum:", sum(nums))
print("all > 0:", all(n > 0 for n in nums))
print("any even:", any(n % 2 == 0 for n in nums))

In [ ]:
# Dataset example: find the longest title (by character length) among the first 200 movies

def longer_title(m1, m2):
    return m1 if len(m1[TITLE_COL]) >= len(m2[TITLE_COL]) else m2

longest = reduce(longer_title, movies[:200])
print("longest title:", longest[TITLE_COL])
print("length:", len(longest[TITLE_COL]))

### Exercise: reduce on the movie dataset

- Using reduce on the dataset:
  - Find the movie with the largest cast
  - (i.e., the movie whose Cast string has the most comma-separated names)
  - Print the title and the number of cast members

In [ ]:
def more_cast(m1, m2):
    # TODO: return whichever movie has more cast members
    cast1 = str(m1[CAST_COL]).split(",")
    cast2 = str(m2[CAST_COL]).split(",")
    return m1 if len(cast1) > len(cast2) else m2

busiest_cast = reduce(more_cast, movies) # reduces to a single value, i.e a single movie record (dict) in our list

if busiest_cast is not None:
    cast_members = str(busiest_cast[CAST_COL]).split(",")
    print("Movie with largest cast:", busiest_cast[TITLE_COL])
    print("Number of cast members:", len(cast_members))

Movie with largest cast: Strange Bedfellows
Number of cast members: 53


## 13) Expressions: comprehensions vs generator expressions

Comprehensions build containers immediately:
- [ ... ] list comprehension
- { ... } set comprehension
- {k: v ...} dict comprehension

Generator expressions use ( ... ) and are lazy:
- they do not build a list
- they produce values as you iterate

A quick rule of thumb:
- if you need all results stored, use a list
- if you just need to loop once, consider a generator

List comprehensions are often more concise and readable for simple transformations, while generator expressions can be more memory efficient for large datasets. List comprehensions are also incredibly powerful and can often be used to replace more complex map/filter/reduce patterns with clearer syntax.

In [ ]:
# Extract all titles as uppercase strings
upper_titles = [m[TITLE_COL].upper() for m in movies[:5]]
print("Uppercase titles:", upper_titles)

In [ ]:
# Get titles of movies released after 2000, formatted with their year
modern_titles = [
    f"{m[TITLE_COL]} ({m[YEAR_COL]})"
    for m in movies
    if int(m[YEAR_COL]) > 2000
]
print("Modern movies count:", len(modern_titles))
print("First 5:", modern_titles[:5])


In [ ]:
# A more complex list comprehension
# Get (title, genre) pairs for all genres of American movies,
# but only where genre is not "unknown"
title_genre_pairs = [
    (m[TITLE_COL], genre.strip())
    for m in movies
    if m[ORIGIN_COL] == "American"
    for genre in str(m[GENRE_COL]).split(",")
    if genre.strip().lower() != "unknown"
]
print("Total (title, genre) pairs:", len(title_genre_pairs))
print("First 5 pairs:", title_genre_pairs[:5])

In [ ]:
# Generator expression: lazily compute plot word counts

# List: builds ALL 100 results immediately in memory
lens_list = [len(m[PLOT_COL].split()) for m in movies[:100]]

# Generator: builds NOTHING yet, waits until you ask
lens_gen = (len(m[PLOT_COL].split()) for m in movies[:100])

print("list type:", type(lens_list))  # <class 'list'>
print("gen  type:", type(lens_gen))   # <class 'generator'>

print("average plot length:", sum(lens_gen) / 100) # computes the generator on the fly, no intermediate list needed

# Now, the generator is exhausted after one pass, so this will return 0. If you want to compute the average again, you would need to create a new generator expression or convert it to a list first.
print("sum again:", sum(lens_gen))    # 0 — already consumed!

In [ ]:
# Same computation in two forms

first_100 = movies[:100]

lens_list = [len(m[PLOT_COL].split()) for m in first_100]
lens_gen = (len(m[PLOT_COL].split()) for m in first_100)

print("list:", type(lens_list), "length:", len(lens_list))
print("gen :", type(lens_gen))

print("sum list:", sum(lens_list))
print("sum gen :", sum(lens_gen))
print("sum gen again:", sum(lens_gen))  # generator is already consumed

### Exercise: List Comprehension

Task

Using a single list comprehension (you may wrap it with `sorted()`), produce a list of strings in this format:

```
"TITLE (YEAR) [Genre1, Genre2]"
```

**Rules:**
1. Only include movies after the 1990s
2. Only include movies where Origin is "American"
3. Exclude any genre that is `"unknown"` or empty after stripping whitespace
4. If a movie has no valid genres after filtering, skip it entirely
5. Each genre must be title-cased (e.g., `"drama"` → `"Drama"`)
6. The final list must be sorted alphabetically by title

In [ ]:
result = sorted(
    [
        f"{m[TITLE_COL]} ({m[YEAR_COL]}) [{', '.join(genres)}]"
        for m in movies
        if m[ORIGIN_COL] == "American"
        and 1990 <= int(m[YEAR_COL])
        if (genres := [ # walrus operator, lets you assign a value to a variable and use that value as part of an expression, all in one step
            g.strip().title()
            for g in str(m[GENRE_COL]).split(",")
            if g.strip() and g.strip().lower() != "unknown"
        ])
    ]
)

print(f"Total matches: {len(result)}")
for r in result[:10]:
    print(r)

Total matches: 5951
 (500) Days of Summer (2009) [Romantic Comedy]
 .45 (2006) [Crime Drama]
 10 MPH (2007) [Documentary]
 10,000 BC (2008) [Adventure]
 102 Dalmatians (2000) [Comedy, Family]
 11:14 (2003) [Crime Drama]
 12/12/12 (2012) [Horror]
 13 Going on 30 (2004) [Comedy]
 13 Moons (2002) [Comedy, Drama]
 15 Minutes (2001) [Action, Crime]


## 15) Dataclasses

A dataclass is a small, lightweight way to define a class whose main job is to hold data.

What dataclasses give you automatically:
- an initializer (__init__)
- a useful string representation (__repr__)
- comparisons (optional)
- default values (optional)

You can also choose to make instances “frozen”:
- frozen=True makes assignment to fields fail after creation
- the intent is to reduce accidental mutation in your code

Why use dataclasses?
- they are more structured than plain dictionaries
- they can have methods for behavior
- they can be immutable if you want

In [ ]:
from dataclasses import dataclass, asdict, replace

@dataclass
class Point:
    x: int
    y: int

p = Point(2, 5)
print(p)
print(asdict(p))

In [ ]:
@dataclass(frozen=True)
class User:
    name: str
    age: int

u = User("Ana", 30)
print(u)

# Uncomment to see what "frozen" means:
# u.age = 31

## 16) Using a dataclass with the movie dataset

We can represent one movie record as a Movie object.
This can make code cleaner than indexing dictionaries with strings.

We will keep the dataclass small:
- year, title, origin, plot

Then we can add simple methods that operate on the fields.

In [ ]:
@dataclass(frozen=True)
class Movie:
    year: int
    title: str
    origin: str
    plot: str

    def plot_word_count(self):
        return len(self.plot.split())

def to_movie(record):
    return Movie(
        year=int(record[YEAR_COL]),
        title=str(record[TITLE_COL]),
        origin=str(record[ORIGIN_COL]),
        plot=str(record[PLOT_COL]),
    )

movies_dc = [to_movie(m) for m in movies[:10]]
print(movies_dc[0])
print("plot words:", movies_dc[0].plot_word_count())

In [ ]:
# replace lets you create a new instance with a change (useful with frozen dataclasses)

m0 = movies_dc[0]
m1 = replace(m0, title=m0.title.upper())

print("original:", m0.title)
print("new     :", m1.title)

### Exercise: dataclasses with the dataset

Task
1) Add a method to Movie called mentions(self, word) that returns True if word is in the plot (case-insensitive).
2) Convert the first 500 records into Movie objects.
3) Filter those Movie objects to keep only movies whose plots mention "love".
4) Print the first 15 matching titles.

Hints
- In the method: use self.plot.lower()
- Use a list comprehension or filter(...)
- Keep the method short

In [ ]:
# TODO: add a method to Movie above, then re-run the Movie class cell.

movies_dc_500 = [to_movie(m) for m in movies[:500]]

# TODO: use your mentions method here
matches = []  # replace with filtering expression

print([m.title for m in matches[:15]])